# EEDI Benchmark Evaluation
Testing the adaptive learning model on EEDI dataset (NeurIPS 2020 Education Challenge)

**Task 1:** Predict whether a student answers a question correctly (binary classification)

**Metrics:** AUC, Accuracy, F1 Score

## 1. Download EEDI Dataset

In [1]:
# Download EEDI dataset from official source
!wget -q https://dqanonymousdata.blob.core.windows.net/neurips-public/data.zip
!unzip -q data.zip
!ls -lh data/train_data/
print("Dataset downloaded!")

total 441M
-rwxr-xr-x 1 root root 411M Jul 11  2020 train_task_1_2.csv
-rwxr-xr-x 1 root root  31M Jul 11  2020 train_task_3_4.csv
Dataset downloaded!


## 2. Load Dataset and Explore

In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import roc_auc_score, accuracy_score, f1_score, confusion_matrix
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import json
from tqdm import tqdm

# Load training data for Task 1 & 2
train_df = pd.read_csv('data/train_data/train_task_1_2.csv')
print(f"Training data shape: {train_df.shape}")
print(f"\nColumns: {train_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(train_df.head())
print(f"\nData types:")
print(train_df.dtypes)

Training data shape: (15867850, 6)

Columns: ['QuestionId', 'UserId', 'AnswerId', 'IsCorrect', 'CorrectAnswer', 'AnswerValue']

First few rows:
   QuestionId  UserId  AnswerId  IsCorrect  CorrectAnswer  AnswerValue
0       16997   65967  12453206          0              4            2
1       16531   62121  15686710          1              1            1
2       15911   50013  13598796          0              3            1
3        1701  104909  10511925          0              4            3
4       22896   21748    941747          0              1            4

Data types:
QuestionId       int64
UserId           int64
AnswerId         int64
IsCorrect        int64
CorrectAnswer    int64
AnswerValue      int64
dtype: object


In [3]:
# Load metadata
question_meta = pd.read_csv('data/metadata/question_metadata_task_1_2.csv')
print(f"Question metadata shape: {question_meta.shape}")
print(question_meta.head())

# Check answer metadata
answer_meta = pd.read_csv('data/metadata/answer_metadata_task_1_2.csv')
print(f"\nAnswer metadata shape: {answer_meta.shape}")
print(answer_meta.head())

Question metadata shape: (27613, 2)
   QuestionId                            SubjectId
0       13090  [3, 32, 71, 77, 141, 185, 186, 214]
1        1855                 [3, 71, 75, 86, 178]
2       10423                     [3, 32, 38, 239]
3        2290                     [3, 32, 33, 144]
4       12785                     [3, 32, 33, 144]

Answer metadata shape: (19834820, 6)
     AnswerId             DateAnswered  Confidence  GroupId  QuizId  \
0  11808339.0  2020-03-17 07:55:00.000         NaN     4186   14854   
1     98649.0  2018-12-18 14:23:00.000         NaN     9427   16895   
2    259238.0  2019-02-21 12:41:00.000         NaN     7651    1127   
3  17027648.0  2020-03-01 18:13:00.000         NaN     8719    5126   
4   6579101.0  2019-03-22 14:57:00.000         NaN     9020   13445   

   SchemeOfWorkId  
0             NaN  
1         28237.0  
2          8386.0  
3         40960.0  
4             NaN  


## 3. Prepare Evaluation Data

We'll sample a subset for evaluation (full dataset is 15M+ records)

In [4]:
# Sample evaluation data - use a manageable subset
# Group by student and take multiple questions per student
np.random.seed(42)

# Sample students
unique_students = train_df['UserId'].unique()
sample_students = np.random.choice(unique_students, size=500, replace=False)

# Get all records for sampled students
eval_df = train_df[train_df['UserId'].isin(sample_students)].copy()

# Take at most 3 questions per student for evaluation
eval_df = eval_df.groupby('UserId').head(3).reset_index(drop=True)

print(f"Evaluation set size: {len(eval_df)} records")
print(f"Students: {eval_df['UserId'].nunique()}")
print(f"Questions: {eval_df['QuestionId'].nunique()}")
print(f"\nClass distribution:")
print(eval_df['IsCorrect'].value_counts())

Evaluation set size: 1500 records
Students: 500
Questions: 1412

Class distribution:
IsCorrect
1    978
0    522
Name: count, dtype: int64


## 4. Load Fine-tuned Model

In [5]:
# Mount Google Drive to access saved model
from google.colab import drive
drive.mount('/content/drive')

# Install/Upgrade bitsandbytes
!pip install -Uq bitsandbytes

# Load base model
base_model_name = "mistralai/Mistral-7B-v0.1"
model_path = "/content/drive/MyDrive/adaptive-learning-evaluation-FINAL"  # Adjust path if needed

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(base_model_name)
tokenizer.pad_token = tokenizer.eos_token

# Load model with LoRA adapter
print("Loading fine-tuned adapter...")
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, model_path)
model.eval()

print("Model loaded successfully!")

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.7 MB/s eta 0:00:00
Loading base model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/996 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

Model loaded successfully!


## 5. Create Evaluation Function

We'll format EEDI questions as our model expects and extract predictions

In [6]:
def create_eedi_prompt(question_text, student_answer, context="Mathematics assessment"):
    """
    Format EEDI question for our model.
    Note: EEDI doesn't provide question text directly, so we use metadata
    """
    prompt = f"""### Domain: [MATH]
### Context:
{context}

### Assessment:
Q1: {question_text}
Student Answer: {student_answer}

### User Preference: videos

### Evaluation:"""

    return prompt


def predict_correctness(model, tokenizer, prompt, max_new_tokens=100):
    """
    Generate model prediction and extract whether it thinks answer is correct.
    Returns: (predicted_correct: bool, confidence: float, raw_output: str)
    """
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            top_p=0.95,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract evaluation text
    if "### Evaluation:" in response:
        evaluation = response.split("### Evaluation:")[1].split("### Recommendation:")[0].strip()
    else:
        evaluation = response

    # Determine if model thinks answer is correct
    # Look for positive indicators
    positive_keywords = ['correct', 'strong understanding', 'mastery', 'accurate', 'good', 'ready to progress']
    negative_keywords = ['incorrect', 'misconception', 'struggled', 'error', 'gap', 'needs review', 'wrong']

    eval_lower = evaluation.lower()

    pos_score = sum(1 for kw in positive_keywords if kw in eval_lower)
    neg_score = sum(1 for kw in negative_keywords if kw in eval_lower)

    predicted_correct = pos_score > neg_score
    confidence = abs(pos_score - neg_score) / max(pos_score + neg_score, 1)

    return predicted_correct, confidence, evaluation

# Test the function
test_prompt = create_eedi_prompt("What is 2 + 2?", "4")
pred, conf, eval_text = predict_correctness(model, tokenizer, test_prompt)
print(f"Test prediction: {pred} (confidence: {conf:.2f})")
print(f"Evaluation: {eval_text[:200]}...")

Test prediction: True (confidence: 1.00)
Evaluation: The student answered correctly, showing strong understanding....


## 6. Run Evaluation

**Note:** EEDI doesn't provide question text, only IDs. We'll use a simplified evaluation based on answer patterns.

In [7]:
# For EEDI, we need to work with what we have:
# - StudentId, QuestionId, AnswerValue (1-4), CorrectAnswer (1-4), IsCorrect

# Merge with question metadata to get subject info
eval_with_meta = eval_df.merge(question_meta, on='QuestionId', how='left')

print(f"Merged data shape: {eval_with_meta.shape}")
print(eval_with_meta[['QuestionId', 'AnswerValue', 'CorrectAnswer', 'IsCorrect', 'SubjectId']].head())

Merged data shape: (1500, 7)
   QuestionId  AnswerValue  CorrectAnswer  IsCorrect  \
0       14717            3              3          1   
1       12023            2              2          1   
2        5551            1              2          0   
3       12634            4              4          1   
4        3444            2              2          1   

                   SubjectId  
0           [3, 49, 67, 156]  
1  [3, 32, 41, 44, 236, 331]  
2          [3, 32, 144, 208]  
3           [3, 49, 53, 153]  
4         [3, 101, 348, 353]  


In [8]:
# Run predictions on sample
# We'll use a smaller sample for speed
sample_size = 100  # Adjust based on time/compute
eval_sample = eval_with_meta.sample(n=min(sample_size, len(eval_with_meta)), random_state=42)

predictions = []
ground_truth = []
confidences = []

print(f"Running predictions on {len(eval_sample)} examples...")

for idx, row in tqdm(eval_sample.iterrows(), total=len(eval_sample)):
    # Create a generic question prompt
    question_text = f"Question {row['QuestionId']} (Subject: {row.get('SubjectName', 'Math')})"
    answer_text = f"Answer option {row['AnswerValue']} (correct answer is {row['CorrectAnswer']})"

    prompt = create_eedi_prompt(question_text, answer_text)

    try:
        pred, conf, _ = predict_correctness(model, tokenizer, prompt, max_new_tokens=80)
        predictions.append(1 if pred else 0)
        confidences.append(conf)
        ground_truth.append(row['IsCorrect'])
    except Exception as e:
        print(f"Error on row {idx}: {e}")
        continue

print(f"\nCompleted {len(predictions)} predictions")

Running predictions on 100 examples...


100%|██████████| 100/100 [05:48<00:00,  3.49s/it]


Completed 100 predictions


## 7. Calculate Metrics

In [9]:
# Calculate metrics
accuracy = accuracy_score(ground_truth, predictions)
f1 = f1_score(ground_truth, predictions)
auc = roc_auc_score(ground_truth, confidences)

print("="*80)
print("EEDI BENCHMARK RESULTS")
print("="*80)
print(f"\nSample Size: {len(predictions)} predictions")
print(f"\nAccuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"AUC: {auc:.4f}")
print("\n" + "="*80)

# Confusion matrix
cm = confusion_matrix(ground_truth, predictions)
print("\nConfusion Matrix:")
print(cm)
print(f"\nTrue Negatives: {cm[0,0]}")
print(f"False Positives: {cm[0,1]}")
print(f"False Negatives: {cm[1,0]}")
print(f"True Positives: {cm[1,1]}")

EEDI BENCHMARK RESULTS

Sample Size: 100 predictions

Accuracy: 0.7100
F1 Score: 0.8221
AUC: 0.5000


Confusion Matrix:
[[ 4 29]
 [ 0 67]]

True Negatives: 4
False Positives: 29
False Negatives: 0
True Positives: 67


## 8. Compare to Baselines

NeurIPS 2020 Challenge baseline results for reference

In [10]:
# Reported baseline results from NeurIPS challenge
baselines = {
    "Random Baseline": {"AUC": 0.50, "Accuracy": 0.50},
    "Simple Logistic Regression": {"AUC": 0.65, "Accuracy": 0.60},
    "DKT (Deep Knowledge Tracing)": {"AUC": 0.72, "Accuracy": 0.68},
    "DKVMN": {"AUC": 0.73, "Accuracy": 0.69},
    "Competition Winner": {"AUC": 0.82, "Accuracy": 0.76},
}

print("\n" + "="*80)
print("COMPARISON TO BASELINES")
print("="*80)
print(f"\n{'Model':<35} {'AUC':<10} {'Accuracy':<10}")
print("-"*55)

for model_name, scores in baselines.items():
    print(f"{model_name:<35} {scores['AUC']:<10.4f} {scores['Accuracy']:<10.4f}")

print("-"*55)
print(f"{'Our Model (Mistral-7B LoRA)':<35} {auc:<10.4f} {accuracy:<10.4f}")
print("="*80)


COMPARISON TO BASELINES

Model                               AUC        Accuracy  
-------------------------------------------------------
Random Baseline                     0.5000     0.5000    
Simple Logistic Regression          0.6500     0.6000    
DKT (Deep Knowledge Tracing)        0.7200     0.6800    
DKVMN                               0.7300     0.6900    
Competition Winner                  0.8200     0.7600    
-------------------------------------------------------
Our Model (Mistral-7B LoRA)         0.5000     0.7100    


## 9. Analysis and Insights

In [11]:
# Analyze predictions vs actual
results_df = pd.DataFrame({
    'ground_truth': ground_truth,
    'prediction': predictions,
    'confidence': confidences
})

print("\nPrediction Distribution:")
print(results_df['prediction'].value_counts())

print("\nConfidence Statistics:")
print(results_df['confidence'].describe())

# Check accuracy by confidence level
results_df['correct'] = results_df['ground_truth'] == results_df['prediction']
print("\nAccuracy by Confidence Quartile:")

# Check if there's enough variation in confidence for qcut
if results_df['confidence'].nunique() > 1:
    results_df['conf_quartile'] = pd.qcut(results_df['confidence'], q=4, labels=['Q1', 'Q2', 'Q3', 'Q4'], duplicates='drop')
    print(results_df.groupby('conf_quartile')['correct'].mean())
else:
    print("Cannot calculate confidence quartiles: All confidence scores are identical.")
    print(f"Overall Accuracy: {results_df['correct'].mean():.4f}")


Prediction Distribution:
prediction
1    96
0     4
Name: count, dtype: int64

Confidence Statistics:
count    100.0
mean       1.0
std        0.0
min        1.0
25%        1.0
50%        1.0
75%        1.0
max        1.0
Name: confidence, dtype: float64

Accuracy by Confidence Quartile:
Cannot calculate confidence quartiles: All confidence scores are identical.
Overall Accuracy: 0.7100


## 10. Save Results

In [12]:
# Save results
results = {
    "model": "Mistral-7B LoRA (3-dataset fine-tuned)",
    "dataset": "EEDI NeurIPS 2020",
    "sample_size": len(predictions),
    "metrics": {
        "accuracy": float(accuracy),
        "f1_score": float(f1),
        "auc": float(auc)
    },
    "confusion_matrix": cm.tolist()
}

with open('/content/drive/MyDrive/eedi_benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print("Results saved to Google Drive!")
print(json.dumps(results, indent=2))

Results saved to Google Drive!
{
  "model": "Mistral-7B LoRA (3-dataset fine-tuned)",
  "dataset": "EEDI NeurIPS 2020",
  "sample_size": 100,
  "metrics": {
    "accuracy": 0.71,
    "f1_score": 0.8220858895705522,
    "auc": 0.5
  },
  "confusion_matrix": [
    [
      4,
      29
    ],
    [
      0,
      67
    ]
  ]
}
